In [8]:
import os
import random
import glob
import json
from PIL import Image, ImageDraw, ImageFont, ImageStat

INPUT_DIR = "radar_backgrounds/images"
LABEL_DIR = "radar_backgrounds/labels"
FONT_DIR = "fonts"
OUTPUT_IMG_DIR = "dataset/images"
OUTPUT_ANN_DIR = "dataset/annotations"

MIN_FONT_SIZE = 10
MAX_FONT_SIZE = 20

LIGHT_TEXT_COLORS = [
    (100, 255, 100),
    (255, 200, 80),
    (150, 230, 255),
    (240, 240, 240),
    (255, 100, 100)
]

DARK_TEXT_COLORS = [
    (0, 0, 0),
    (0, 50, 0),
    (0, 0, 100),
    (50, 50, 50)
]

os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_ANN_DIR, exist_ok=True)

def load_fonts():
    fonts = glob.glob(os.path.join(FONT_DIR, "*.[ot]tf"))
    if not fonts:
        print("Шрифти не знайдено! Використовую стандартний.")
        return ["arial.ttf"]
    return fonts

def get_contrasting_color(image_crop):
    grayscale = image_crop.convert("L")
    stat = ImageStat.Stat(grayscale)
    avg_luminance = stat.mean[0]
    if avg_luminance > 130:
        return random.choice(DARK_TEXT_COLORS)
    else:
        return random.choice(LIGHT_TEXT_COLORS)

def generate_radar_data():
    mode = random.choice(["polar", "cartesian", "no h"])

    if mode == "polar":
        az = f"{random.uniform(0, 359.9):.1f}"
        rng = f"{random.uniform(50, 5000):.1f}"
        return [("Az:", az), ("Rg:", rng)]

    elif mode == "cartesian":
        x = f"{random.randint(-9999, 9999)}"
        y = f"{random.randint(-9999, 9999)}"
        h = f"{random.randint(0, 3000)}"
        return [("X:", x), ("Y:", y), ("H:", h)]

    else:
        x = f"{random.uniform(400000, 600000):.2f}"
        y = f"{random.uniform(4000000, 6000000):.2f}"
        return [("X:", x), ("Y:", y)]

def try_fit_text(draw, font_path, box_w, box_h, data):
    prefer_row = box_w > (box_h * 2.2)
    for size in range(MAX_FONT_SIZE, MIN_FONT_SIZE - 1, -1):
        try:
            font = ImageFont.truetype(font_path, size)
        except:
            font = ImageFont.load_default()

        ascent, descent = font.getmetrics()
        lh = ascent + descent

        if prefer_row:
            full_line_str = ""
            for t, v in data:
                full_line_str += f"{t} {v}   "
            full_line_str = full_line_str.strip()

            w = font.getlength(full_line_str)
            if w <= box_w and lh <= box_h:
                return font, lh, [data], (box_h - lh) / 2

        total_h = lh * len(data)
        max_w = 0
        for t, v in data:
            line_str = f"{t} {v}"
            w = font.getlength(line_str)
            if w > max_w: max_w = w

        if total_h <= box_h and max_w <= box_w:
            vertical_lines = [[item] for item in data]
            return font, lh, vertical_lines, (box_h - total_h) / 2

    return None, None, None, None

def process_images():
    images = glob.glob(os.path.join(INPUT_DIR, "*.*"))
    fonts = load_fonts()

    print(f"Знайдено {len(images)} зображень. Генерація...")
    count = 0

    for img_path in images:
        filename = os.path.basename(img_path)
        name_no_ext = os.path.splitext(filename)[0]

        label_path = os.path.join(LABEL_DIR, name_no_ext + ".txt")
        if not os.path.exists(label_path):
            continue

        zones = []
        with open(label_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                if len(parts) == 4:
                    zones.append(list(map(int, parts)))

        if not zones: continue

        try:
            original_img = Image.open(img_path).convert("RGB")
        except:
            continue

        for v_idx in range(3):
            img_copy = original_img.copy()
            draw = ImageDraw.Draw(img_copy)
            file_annotations = []

            for zone in zones:
                x1, y1, x2, y2 = zone
                bw, bh = x2 - x1, y2 - y1

                if bw < 5 or bh < 5: continue

                data = generate_radar_data()
                font_path = random.choice(fonts)

                font, lh, lines_structure, start_y = try_fit_text(draw, font_path, bw, bh, data)

                if font:
                    zone_crop = original_img.crop((x1, y1, x2, y2))
                    text_color = get_contrasting_color(zone_crop)

                    current_y = y1 + start_y
                    chars_in_zone = []

                    for line_items in lines_structure:
                        full_str = ""
                        for t, v in line_items:
                            full_str += f"{t} {v}   "
                        full_str = full_str.strip()

                        line_w = font.getlength(full_str)
                        current_x = x1 + (bw - line_w) / 2

                        for t, v in line_items:
                            for char in t:
                                draw.text((current_x, current_y), char, font=font, fill=text_color)
                                cw = font.getlength(char)
                                chars_in_zone.append({
                                    "char": char, "class": "tag",
                                    "bbox": [current_x, current_y, current_x+cw, current_y+lh]
                                })
                                current_x += cw

                            space_w = font.getlength(" ")
                            current_x += space_w
                            for char in v:
                                cls = "digit"
                                if char == "-": cls = "minus"
                                elif char == ".": cls = "point"
                                elif not char.isdigit(): cls = "tag"

                                draw.text((current_x, current_y), char, font=font, fill=text_color)
                                cw = font.getlength(char)
                                chars_in_zone.append({
                                    "char": char, "class": cls,
                                    "bbox": [current_x, current_y, current_x+cw, current_y+lh]
                                })
                                current_x += cw

                            current_x += (space_w * 3)

                        current_y += lh

                    file_annotations.append({
                        "zone": zone,
                        "data_raw": data,
                        "chars": chars_in_zone
                    })

            if file_annotations:
                new_fname = f"{name_no_ext}_v{v_idx}"
                img_copy.save(os.path.join(OUTPUT_IMG_DIR, new_fname + ".jpg"))
                with open(os.path.join(OUTPUT_ANN_DIR, new_fname + ".json"), 'w') as f:
                    json.dump(file_annotations, f, indent=4)
                count += 1

    print(f"Готово! Згенеровано {count} зображень.")

if __name__ == "__main__":
    process_images()

Знайдено 117 зображень. Генерація...
Готово! Згенеровано 351 зображень.
